<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026/%D0%A7%D0%B0%D1%81%D1%82%D1%8C_V_%D0%9D%D0%B5%D0%BB%D0%B8%D0%BD%D0%B5%D0%B9%D0%BD%D1%8B%D0%B5_%D0%BD%D0%B5%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 7. Решающие деревья (CART): теория, критерии расщепления и вероятностный смысл

Линейные модели накладывают жёсткую параметрическую форму на зависимость. Решающие деревья предлагают принципиально иной подход — они рекурсивно разбивают пространство признаков на подобласти, в каждой из которых предсказание просто. Эта идея приводит к кусочно‑постоянным функциям, способным улавливать сложные нелинейные закономерности без предположений о глобальной структуре.

### 1. Геометрическая и функциональная интерпретация

Решающее дерево задаёт разбиение входного пространства $\mathbb{R}^D$ на множество непересекающихся гиперпрямоугольников, грани которых параллельны осям координат. Каждая такая область $R_m \subseteq \mathbb{R}^D$ ассоциирована с листом дерева, и модель предсказывает постоянное значение $c_m$ для всех наблюдений, попавших в $R_m$:

$$
f(\mathbf{x}) = \sum_{m=1}^{M} c_m \cdot \mathbf{1}_{\{\mathbf{x} \in R_m\}},
$$

где $M$ — число листьев, а $\mathbf{1}_{\{\cdot\}}$ — индикаторная функция. Таким образом, дерево реализует кусочно‑постоянную аппроксимацию неизвестной целевой зависимости.

Структура разбиения задаётся **рекурсивным бинарным деревом**: каждый внутренний узел содержит условие вида «$x_j \le \theta$», где $x_j$ — один из признаков, а $\theta$ — пороговое значение. Объекты, удовлетворяющие условию, направляются в левое поддерево, остальные — в правое. Процесс рекурсивно продолжается до тех пор, пока не сработает критерий остановки, после чего узел становится листом. Геометрически каждое расщепление проводит границу, параллельную координатным осям, поэтому результирующие области являются объединением многомерных прямоугольников со сторонами, параллельными осям. Это свойство одновременно обеспечивает высокую интерпретируемость: любое решение дерева можно представить как набор правил «если … то …».

### 2. Жадное построение дерева для регрессии

Пусть обучающая выборка состоит из пар $(\mathbf{x}_i, y_i) \in \mathbb{R}^D \times \mathbb{R}$, $i = 1,\dots,n$. В узле $t$, содержащем $m$ наблюдений, ищется расщепление по признаку $j$ и порогу $\theta$, минимизирующее сумму квадратов ошибок в двух образовавшихся дочерних узлах:

$$
Q(j, \theta) = \min_{c_1} \sum_{i \in R_1(j,\theta)} (y_i - c_1)^2 + \min_{c_2} \sum_{i \in R_2(j,\theta)} (y_i - c_2)^2,
$$

где
$$
R_1(j,\theta) = \{i \mid x_{ij} \le \theta\}, \qquad R_2(j,\theta) = \{i \mid x_{ij} > \theta\}.
$$

Оптимальные значения констант $c_1$ и $c_2$, минимизирующие каждое из слагаемых, суть выборочные средние целевой переменной в соответствующих регионах:

$$
c_1 = \frac{1}{m_L} \sum_{i \in R_1} y_i, \qquad c_2 = \frac{1}{m_R} \sum_{i \in R_2} y_i,
$$

где $m_L = |R_1|$, $m_R = |R_2|$, $m_L + m_R = m$. После подстановки этих значений критерий приобретает эквивалентную форму — максимизацию **уменьшения дисперсии**:

$$
\Delta = \sigma^2_{\text{parent}} - \frac{m_L}{m}\sigma^2_L - \frac{m_R}{m}\sigma^2_R,
$$

где $\sigma^2_{\text{node}} = \frac{1}{m_{\text{node}}} \sum_{i \in \text{node}} (y_i - \bar{y}_{\text{node}})^2$ — дисперсия отклика в узле. Иными словами, на каждом шаге алгоритм выбирает то расщепление, которое сильнее всего уменьшает разброс целевой переменной. Для непрерывного признака $j$ перебор всех возможных порогов осуществляется сортировкой его уникальных значений; вычислительная сложность одного такого поиска составляет $O(m \log m)$.

### 3. Критерии impurity для классификации

В задаче классификации с $K$ классами целевая переменная принимает значения $y_i \in \{1,\dots,K\}$. В каждом узле $t$, содержащем $m$ наблюдений, оценивается распределение классов:

$$
\hat{p}_k = \frac{m_k}{m}, \qquad m_k = |\{i \in t: y_i = k\}|.
$$

Для измерения «загрязнённости» узла используется **impurity-функция** $I(\hat{p}_1,\dots,\hat{p}_K)$, удовлетворяющая требованиям:
- $I \ge 0$, причём $I = 0$ тогда и только тогда, когда все объекты узла принадлежат одному классу;
- $I$ достигает максимума при равномерном распределении $\hat{p}_k = 1/K$;
- $I$ — симметричная и вогнутая функция вектора вероятностей, что гарантирует невозрастание impurity при расщеплении.

Стандартные меры impurity:

* **Ошибка классификации (misclassification error):**
  $$ I_{\text{err}} = 1 - \max_k \hat{p}_k. $$
  Она непосредственно равна доле неверных предсказаний, если в узле предсказывается наиболее частый класс.

* **Энтропия:**
  $$ I_{\text{ent}} = -\sum_{k=1}^{K} \hat{p}_k \log_2 \hat{p}_k. $$
  Энтропия измеряет среднее количество информации (в битах), необходимое для описания класса случайно выбранного объекта из узла; нулевая энтропия означает полную определённость.

* **Индекс Джини (Gini impurity):**
  $$ I_{\text{Gini}} = \sum_{k=1}^{K} \hat{p}_k (1 - \hat{p}_k) = 1 - \sum_{k=1}^{K} \hat{p}_k^2. $$
  Его можно интерпретировать как вероятность ошибки, если классифицировать объект случайным образом, выбирая класс с вероятностями $\hat{p}_k$ (или, эквивалентно, вероятность того, что два случайно выбранных объекта узла принадлежат разным классам).

Все три функции являются вогнутыми по $\hat{\mathbf{p}}$ (энтропия и индекс Джини — строго вогнуты), достигают минимума $0$ на вырожденных распределениях и максимума при равномерном распределении.

Расщепление узла выбирается так, чтобы максимизировать **информационный выигрыш (Information Gain)** — взвешенное уменьшение impurity:

$$
\text{Gain} = I_{\text{parent}} - \frac{m_L}{m} I_L - \frac{m_R}{m} I_R.
$$

Максимизация Gain эквивалентна минимизации средневзвешенной impurity дочерних узлов, что заставляет листья быть как можно более «чистыми».

#### Углублённый взгляд: связь impurity с вероятностными моделями и байесовским риском

Минимизация энтропии в дочерних узлах эквивалентна максимизации **взаимной информации** между признаками и классом. Пусть $Y$ — случайная величина класса, а $S$ — индикатор попадания в левый или правый дочерний узел. Тогда условная энтропия $H(Y \mid S)$ есть в точности взвешенная сумма энтропий $\frac{m_L}{m} H(Y \mid S=L) + \frac{m_R}{m} H(Y \mid S=R)$. Поскольку безусловная энтропия $H(Y)$ (энтропия родительского узла) фиксирована, уменьшение энтропии равно взаимной информации $I(Y; S)$ — мере того, насколько знание расщепления снижает неопределённость о классе.

Индекс Джини допускает аналогичную интерпретацию в терминах среднеквадратичной ошибки вероятностного прогноза. Пусть истинная принадлежность к классу кодируется one-hot вектором $\mathbf{y} \in \{0,1\}^K$, а прогнозные вероятности в узле — $\hat{\mathbf{p}}$. Тогда квадрат евклидовой ошибки составляет $\|\mathbf{y} - \hat{\mathbf{p}}\|^2 = \sum_{k=1}^{K} (y_k - \hat{p}_k)^2$. Математическое ожидание этой величины по истинному распределению классов $\hat{\mathbf{p}}$ даёт

$$
\mathbb{E}\bigl[ \|\mathbf{y} - \hat{\mathbf{p}}\|^2 \bigr]
= \sum_{j=1}^{K} \hat{p}_j \Bigl( (1 - \hat{p}_j)^2 + \sum_{k \ne j} \hat{p}_k^2 \Bigr)
= 1 - \sum_{k=1}^{K} \hat{p}_k^2 = I_{\text{Gini}}.
$$

Таким образом, индекс Джини есть ожидаемая квадратичная ошибка (мультиклассовый Brier score) внутри узла при прогнозе вероятностей классов, равных относительным частотам. Жадное уменьшение индекса Джини, следовательно, соответствует минимизации среднеквадратичной ошибки предсказанных вероятностей классов.

Байесовский риск (ожидаемая вероятность ошибки) при оптимальном байесовском классификаторе в узле равен $1 - \max_k \hat{p}_k$, то есть $I_{\text{err}}$. Энтропия и индекс Джини являются вогнутыми верхними границами для этой величины, поэтому их минимизация косвенно снижает и долю неверных классификаций. При этом они лучше различают «почти чистые» и «совершенно чистые» узлы, чем сама ошибка классификации, что делает процесс обучения более стабильным и чувствительным к улучшениям.

### 4. Почему impurity-меры предпочитают многозначные признаки?

И энтропия, и индекс Джини обладают скрытым смещением в пользу признаков с большим числом уникальных значений. Рассмотрим крайний случай: признак «уникальный идентификатор объекта» позволяет построить расщепление, при котором каждый дочерний узел содержит ровно одно наблюдение. Тогда impurity листьев обращается в нуль, информационный выигрыш достигает максимума, но такое расщепление абсолютно бесполезно для обобщения.

Для компенсации этого эффекта алгоритм C4.5 вместо обычного Information Gain использует **Information Gain Ratio** — информационный выигрыш, нормированный на собственную энтропию расщепления (Split Information):

$$
\text{GainRatio} = \frac{\text{Gain}}{-\sum_{v} \frac{m_v}{m} \log_2 \frac{m_v}{m}},
$$

где сумма берётся по всем дочерним узлам $v$. Знаменатель измеряет, насколько сильно расщепление само по себе сегментирует данные: чем больше уникальных значений у признака, тем выше эта энтропия, и GainRatio штрафует такие признаки. В алгоритме CART, обычно применяемом в современных реализациях, эта проблема менее выражена благодаря бинарной природе расщеплений; для категориальных признаков также используют специальные методы объединения категорий, что дополнительно снижает неоправданное дробление.

### 5. Проблема переобучения и регуляризация

Решающее дерево, не имеющее ограничений на рост, способно идеально воспроизвести обучающую выборку — достаточно построить дерево, в котором каждый лист содержит ровно один объект. При этом обучающая ошибка станет нулевой, однако дисперсия модели окажется чрезвычайно высокой, и качество на новых данных резко ухудшится.

Для борьбы с переобучением применяют стратегии регуляризации, разделяемые на **pre‑pruning** (раннюю остановку) и **post‑pruning** (отсечение ветвей после построения полного дерева).

**Pre‑pruning** вводит критерии, запрещающие дальнейшее разбиение узла, если:
- число объектов в узле меньше `min_samples_split`;
- хотя бы один из дочерних узлов получит меньше `min_samples_leaf` объектов;
- достигнута максимальная глубина `max_depth`;
- улучшение impurity меньше заданного порога `min_impurity_decrease`.

Эти параметры непосредственно ограничивают сложность дерева.

**Cost‑complexity pruning** (используемый в оригинальном алгоритме CART) строит последовательность вложенных поддеревьев, минимизируя штрафную функцию:

$$
R_\alpha(T) = R(T) + \alpha |T|,
$$

где $R(T)$ — суммарная ошибка дерева на обучающей выборке (например, MSE для регрессии или число неверных классификаций для классификации), $|T|$ — количество листьев, а $\alpha \ge 0$ — параметр, балансирующий точность и сложность. При $\alpha = 0$ оптимальным является самое глубокое дерево; с ростом $\alpha$ более выгодными становятся поддеревья меньшего размера. Алгоритм эффективно находит всю последовательность оптимальных поддеревьев для всех возможных значений $\alpha$ путём последовательного отсечения наименее «важных» ветвей. Оптимальное значение $\alpha$ выбирается с помощью кросс‑валидации, что даёт дерево с хорошим балансом между смещением и дисперсией.

---

### Вопросы для самопроверки

#### Для начинающих
1. Что такое impurity-мера в контексте решающих деревьев и каким требованиям она должна удовлетворять?
2. Почему в качестве критерия расщепления при классификации предпочтительнее использовать энтропию или индекс Джини, а не непосредственно ошибку классификации?
3. Зачем нужна регуляризация (pruning) деревьев? В чём разница между pre‑pruning и post‑pruning?
4. Как работает критерий «минимальное число объектов в листе» (`min_samples_leaf`) и к каким последствиям для структуры дерева он приводит?
5. Что такое Information Gain Ratio и для чего он был введён? В каком алгоритме он применяется?

#### Для профессионалов
1. Докажите строгую вогнутость энтропии и индекса Джини как функций вектора вероятностей $\hat{\mathbf{p}}$ и покажите, что они достигают глобального максимума при равномерном распределении.
2. Объясните, почему индекс Джини численно равен ожидаемой ошибке классификации, если решение о классе принимается случайным выбором по распределению $\hat{\mathbf{p}}$. Как это связано с Brier score для мультиклассового прогноза вероятностей?
3. Выведите градиент энтропии по предсказанным вероятностям классов в листе $\partial I_{\text{ent}} / \partial \hat{p}_k$ и проинтерпретируйте его в контексте того, как изменение частоты класса влияет на impurity.
4. Обсудите NP‑полноту задачи построения оптимального (минимального по числу листьев или ошибке) решающего дерева для заданной обучающей выборки. Какие следствия это имеет для алгоритмов обучения деревьев на практике?

## Реализация решающего дерева с нуля и анализ сложности

Теоретические принципы, изложенные в предыдущем разделе, допускают непосредственную алгоритмическую реализацию. Ниже мы создадим с нуля регрессионное и классификационное решающие деревья, руководствуясь жадной стратегией CART, и проанализируем их вычислительные характеристики. Все эксперименты выполняются с использованием единого набора импортов.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor as SKDecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier as SKDecisionTreeClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
```

### 1. Структура узла и рекурсивное построение

Узел дерева хранит информацию о расщеплении либо о предсказании, если он является листом. Определим простой класс `Node`:

```python
class Node:
    def __init__(self, feature_index=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature_index = feature_index  # индекс признака для расщепления
        self.threshold = threshold          # порог расщепления
        self.left = left                    # левое поддерево (x <= threshold)
        self.right = right                  # правое поддерево (x > threshold)
        self.value = value                  # предсказание (для листа)
```

Построение дерева выполняется рекурсивной функцией `_build_tree(X, y, depth)`. На каждом шаге проверяются критерии остановки: достигнута максимальная глубина `max_depth`, число объектов в узле меньше `min_samples_split` или все целевые значения одинаковы. Если критерии срабатывают, создаётся лист с предсказанием, равным среднему (для регрессии) или наиболее частому классу (для классификации).

Оптимальное расщепление ищется полным перебором по всем признакам и всем возможным порогам. Для непрерывного признака $j$ значения сортируются, а пороги выбираются как середины между соседними уникальными значениями. Для каждого кандидата вычисляется уменьшение impurity (MSE для регрессии, Gini для классификации), после чего выбирается пара $(j, \theta)$, дающая наилучшее значение критерия. Дочерние узлы формируются рекурсивно на левой ($x_j \le \theta$) и правой ($x_j > \theta$) подвыборках.

### 2. Вычислительная сложность перебора

В одном узле с $m$ объектами и $D$ признаками перебор включает сортировку каждого признака: $O(D \, m \log m)$. После сортировки вычисление impurity для каждого из $O(m)$ порогов можно выполнить за $O(1)$ благодаря накопленным суммам. Следовательно, временные затраты на узел составляют $O(D \, m \log m)$. В сбалансированном дереве глубина $O(\log n)$, и суммарная сложность построения оценивается как $O(D \, n \log^2 n)$ из-за необходимости сортировки на каждом уровне. На практике эта оценка может ухудшаться до $O(D \, n^2 \log n)$ для крайне несбалансированных деревьев, но в среднем при ограничении глубины сложность остаётся приемлемой.

Эффективные библиотеки (например, `sklearn`) снижают накладные расходы путём предварительной сортировки всех признаков на этапе обучения и использования гистограммных приближений для больших данных, что позволяет достичь почти линейной сложности.

### 3. Реализация классов

#### Регрессионное дерево

```python
class DecisionTreeRegressor:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]
        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) \
           or n_samples < self.min_samples_split \
           or np.all(y == y[0]):
            return Node(value=np.mean(y))

        best_mse = float('inf')
        best_feature, best_threshold = None, None
        best_left_idx, best_right_idx = None, None

        for j in range(self.n_features_):
            # Сортировка признака
            sorted_idx = np.argsort(X[:, j])
            X_sorted = X[sorted_idx, j]
            y_sorted = y[sorted_idx]
            # Уникальные значения для порогов (середины)
            thresholds = (X_sorted[:-1] + X_sorted[1:]) / 2.0

            # Накопленные суммы для быстрого пересчёта MSE
            cumsum_y = np.cumsum(y_sorted)
            cumsum_y2 = np.cumsum(y_sorted ** 2)
            n_total = n_samples

            for i, th in enumerate(thresholds):
                left_idx = sorted_idx[:i+1]
                right_idx = sorted_idx[i+1:]
                n_left = i + 1
                n_right = n_total - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                # MSE в левом и правом узлах
                mse_left = np.var(y_sorted[:i+1]) * n_left  # сумма квадратов отклонений
                mse_right = np.var(y_sorted[i+1:]) * n_right
                total_mse = mse_left + mse_right

                if total_mse < best_mse:
                    best_mse = total_mse
                    best_feature = j
                    best_threshold = th
                    best_left_idx = left_idx
                    best_right_idx = right_idx

        if best_feature is None:
            return Node(value=np.mean(y))

        left_subtree = self._build_tree(X[best_left_idx], y[best_left_idx], depth+1)
        right_subtree = self._build_tree(X[best_right_idx], y[best_right_idx], depth+1)
        return Node(feature_index=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])
```

В коде для эффективного вычисления MSE используется свойство, что сумма квадратов отклонений в группе равна $n \cdot \sigma^2$, где $\sigma^2$ — дисперсия. Вместо явного хранения дисперсии мы вычисляем её через кумулятивные суммы.

#### Классификационное дерево

```python
class DecisionTreeClassifier:
    def __init__(self, criterion='gini', max_depth=None,
                 min_samples_split=2, min_samples_leaf=1):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.root = self._build_tree(X, y, depth=0)
        return self

    def _impurity(self, y):
        # Вычисление impurity для вектора меток y
        if len(y) == 0:
            return 0
        counts = np.bincount(y, minlength=self.n_classes_)
        p = counts / len(y)
        if self.criterion == 'gini':
            return 1 - np.sum(p ** 2)
        elif self.criterion == 'entropy':
            # избегаем log(0)
            p = p[p > 0]
            return -np.sum(p * np.log2(p))
        else:
            raise ValueError("criterion must be 'gini' or 'entropy'")

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]
        if (self.max_depth is not None and depth >= self.max_depth) \
           or n_samples < self.min_samples_split \
           or len(np.unique(y)) == 1:
            # лист: наиболее частый класс
            counts = np.bincount(y, minlength=self.n_classes_)
            return Node(value=np.argmax(counts))

        parent_impurity = self._impurity(y)
        best_gain = 0.0
        best_feature, best_threshold = None, None
        best_left_idx, best_right_idx = None, None

        for j in range(self.n_features_):
            sorted_idx = np.argsort(X[:, j])
            X_sorted = X[sorted_idx, j]
            y_sorted = y[sorted_idx]
            thresholds = (X_sorted[:-1] + X_sorted[1:]) / 2.0

            for i, th in enumerate(thresholds):
                left_idx = sorted_idx[:i+1]
                right_idx = sorted_idx[i+1:]
                n_left = i + 1
                n_right = n_samples - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue

                left_impurity = self._impurity(y_sorted[:i+1])
                right_impurity = self._impurity(y_sorted[i+1:])
                weighted_imp = (n_left * left_impurity + n_right * right_impurity) / n_samples
                gain = parent_impurity - weighted_imp

                if gain > best_gain:
                    best_gain = gain
                    best_feature = j
                    best_threshold = th
                    best_left_idx = left_idx
                    best_right_idx = right_idx

        if best_feature is None:
            counts = np.bincount(y, minlength=self.n_classes_)
            return Node(value=np.argmax(counts))

        left_subtree = self._build_tree(X[best_left_idx], y[best_left_idx], depth+1)
        right_subtree = self._build_tree(X[best_right_idx], y[best_right_idx], depth+1)
        return Node(feature_index=best_feature, threshold=best_threshold,
                    left=left_subtree, right=right_subtree)

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])
```

Оба класса следуют идентичной логике построения дерева, отличаясь лишь критерием расщепления и способом вычисления предсказания в листе. Критерий `gini` используется по умолчанию как более вычислительно эффективный.

### 4. Проверка на синтетических данных

#### Регрессия: $y = x^2 + \sin(5x) + \varepsilon$

```python
np.random.seed(42)
n = 100
x = np.linspace(-2, 2, n)
y = x**2 + np.sin(5*x) + np.random.normal(0, 0.3, n)
X = x.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
```

Построим наше дерево с различной глубиной и сравним с библиотечной реализацией.

```python
depths = [2, 5, 10]
plt.figure(figsize=(15, 5))
for i, d in enumerate(depths):
    our_tree = DecisionTreeRegressor(max_depth=d)
    our_tree.fit(X_train, y_train)
    y_pred_our = our_tree.predict(X_test)

    sk_tree = SKDecisionTreeRegressor(max_depth=d, random_state=0)
    sk_tree.fit(X_train, y_train)
    y_pred_sk = sk_tree.predict(X_test)

    x_plot = np.linspace(-2.2, 2.2, 200).reshape(-1,1)
    plt.subplot(1, 3, i+1)
    plt.scatter(X_train, y_train, alpha=0.5, label='train')
    plt.plot(x_plot, our_tree.predict(x_plot), 'r-', label='our')
    plt.plot(x_plot, sk_tree.predict(x_plot), 'g--', label='sklearn')
    plt.title(f'max_depth={d}')
    plt.legend()
plt.tight_layout()
plt.show()
```

Визуализация подтверждает совпадение предсказаний нашей и библиотечной моделей. Дерево малой глубины даёт грубую кусочно‑постоянную аппроксимацию, с увеличением глубины детализация возрастает, но появляется риск переобучения.

#### Классификация: датасет «луны» (make_moons)

```python
X, y = make_moons(n_samples=200, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

plt.figure(figsize=(15, 5))
for i, d in enumerate([1, 3, None]):  # None означает полное дерево
    tree = DecisionTreeClassifier(max_depth=d)
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Построение сетки для контурного графика
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                           np.linspace(x2_min, x2_max, 200))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    Z = tree.predict(grid).reshape(xx1.shape)

    plt.subplot(1, 3, i+1)
    plt.contourf(xx1, xx2, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor='k', cmap='RdYlBu')
    title = f'max_depth={d}' if d is not None else 'без ограничения'
    plt.title(f'{title} (acc={acc:.2f})')
plt.tight_layout()
plt.show()
```

Граница решений с ростом глубины становится всё более изрезанной, полностью адаптируясь к обучающим точкам при отсутствии ограничений. Это наглядно иллюстрирует компромисс между смещением и дисперсией.

### 5. Анализ переобучения и роль гиперпараметров

Для количественной оценки влияния гиперпараметров построим кривые обучения при изменении `max_depth`. Используем тот же регрессионный датасет.

```python
max_depths = np.arange(1, 21)
train_mse, test_mse = [], []
for d in max_depths:
    tree = DecisionTreeRegressor(max_depth=d)
    tree.fit(X_train, y_train)
    train_mse.append(mean_squared_error(y_train, tree.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, tree.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(max_depths, train_mse, 'bo-', label='train MSE')
plt.plot(max_depths, test_mse, 'ro-', label='test MSE')
plt.xlabel('max_depth')
plt.ylabel('MSE')
plt.legend()
plt.grid()
plt.title('Переобучение решающего дерева')
plt.show()
```

График демонстрирует, что с увеличением `max_depth` ошибка на обучении монотонно убывает, вплоть до нуля, тогда как тестовая ошибка сначала снижается, а затем начинает расти — классический признак переобучения. Другие гиперпараметры (`min_samples_leaf`, `min_samples_split`) аналогично ограничивают рост дерева и предотвращают излишнюю подгонку под шум.

---

### Вопросы для самопроверки

#### Для начинающих
1. Как в коде осуществляется выбор порога для расщепления? Почему пороги берутся как середины между соседними уникальными значениями признака?
2. Почему дерево без ограничений (max_depth=None, min_samples_leaf=1) практически всегда переобучается?
3. Каким образом гиперпараметр `min_samples_leaf` влияет на сложность дерева и его способность к обобщению?
4. Как визуализировать границу решений классификатора и что означает её сильная изрезанность?

#### Для профессионалов
1. Объясните, почему сложность наивного перебора порогов составляет $O(D \, m \log m)$ на один узел, и предложите способы её снижения с помощью предварительной сортировки признаков один раз для всего обучения, а не на каждом узле.
2. Сравните жадный алгоритм CART с поиском глобально оптимального дерева минимального размера. Почему последняя задача NP‑полна? К каким практическим последствиям это приводит?
3. При наличии повторяющихся значений признака выбор порога как середины может приводить к вырожденным ситуациям (например, порог равен значению признака). Как обеспечить численную устойчивость и корректность разбиения в таких случаях?

## 8. Бэггинг (Bootstrap Aggregating)

Одиночные решающие деревья страдают от высокой дисперсии: небольшие изменения в обучающей выборке способны радикально перестроить структуру дерева. Бэггинг (Bootstrap Aggregating), предложенный Брейманом (Breiman, 1996), уменьшает дисперсию предсказаний за счёт усреднения большого числа деревьев, обученных на случайно порождённых версиях исходных данных. Этот раздел посвящён математическим основаниям бэггинга, его свойствам и практической реализации.

### 1. Идея бэггинга

Пусть имеется обучающая выборка $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$. Бэггинг строит ансамбль из $B$ базовых моделей (как правило, решающих деревьев), каждая из которых обучается на **бутстреп‑выборке** $\mathcal{D}_b^*$, полученной из $\mathcal{D}$ случайным выбором $n$ объектов **с возвращением**. Таким образом, в каждой бутстреп‑выборке некоторые объекты исходной выборки могут присутствовать несколько раз, а некоторые — отсутствовать. Вероятность для конкретного объекта не попасть в $\mathcal{D}_b^*$ равна

$$
\left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0{,}368.
$$

Следовательно, каждая бутстреп‑выборка содержит в среднем около $63{,}2\%$ уникальных объектов исходной выборки.

Ансамблевое предсказание для регрессии строится как среднее арифметическое предсказаний отдельных моделей:

$$
\hat{f}_{\text{bag}}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(\mathbf{x}),
$$

где $\hat{f}_b$ — модель, обученная на $\mathcal{D}_b^*$. В задаче классификации агрегирование обычно проводят голосованием: большинством (hard voting) или усреднением предсказанных вероятностей классов (soft voting).

### 2. Математическое обоснование снижения дисперсии

Эффективность бэггинга основана на уменьшении дисперсии предсказаний за счёт усреднения. Рассмотрим идеализированную ситуацию: имеется $B$ случайных величин $\hat{f}_1(\mathbf{x}),\dots,\hat{f}_B(\mathbf{x})$, представляющих предсказания моделей ансамбля для фиксированной точки $\mathbf{x}$. Предположим, что все они имеют одинаковое математическое ожидание $\mathbb{E}[\hat{f}_b] = \mu$, одинаковую дисперсию $\mathrm{Var}(\hat{f}_b) = \sigma^2$ и попарную корреляцию $\rho = \mathrm{Corr}(\hat{f}_i, \hat{f}_j)$ для $i \neq j$. Тогда для среднего $\bar{f} = \frac{1}{B}\sum_{b=1}^B \hat{f}_b$ имеем:

$$
\begin{aligned}
\mathrm{Var}(\bar{f})
&= \frac{1}{B^2} \left( \sum_{i=1}^{B} \mathrm{Var}(\hat{f}_i) + \sum_{i \neq j} \mathrm{Cov}(\hat{f}_i, \hat{f}_j) \right) \\
&= \frac{1}{B^2} \bigl( B\sigma^2 + B(B-1)\rho\sigma^2 \bigr) \\
&= \frac{1}{B}\sigma^2 + \frac{B-1}{B}\rho\sigma^2 \\
&= \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
\end{aligned}
$$

При $\rho < 1$ дисперсия среднего строго меньше дисперсии одной модели: $\mathrm{Var}(\bar{f}) < \sigma^2$. С ростом $B$ второе слагаемое стремится к нулю, и дисперсия ансамбля асимптотически приближается к $\rho\sigma^2$. Таким образом, чем меньше корреляция между моделями, тем большее снижение дисперсии достигается; при $\rho = 0$ дисперсия уменьшается в $B$ раз.

#### Углублённый взгляд: асимптотическая дисперсия бэггинга и устойчивость алгоритма

Бутстреп‑ансамбль порождает модели, корреляция $\rho$ между которыми определяется **устойчивостью** базового алгоритма обучения. Для неустойчивых алгоритмов, таких как глубокие решающие деревья, малые изменения обучающей выборки (бутстреп) приводят к значительным вариациям предсказаний, поэтому корреляция между предсказаниями двух деревьев, обученных на разных бутстреп‑выборках, относительно невелика. Напротив, для устойчивых алгоритмов (например, $k$-ближайших соседей с большим $k$) бутстреп‑выборки почти не меняют модель, $\rho \approx 1$, и бэггинг не даёт выигрыша.

Формально, можно показать (см. Efron, Tibshirani, 1993; Buja, Stuetzle, 2006), что асимптотическая дисперсия бэггинг‑оценки связана с функцией влияния базового алгоритма. Для деревьев функция влияния неограничена, что и объясняет высокую эффективность бэггинга.

Бэггинг практически не изменяет смещение модели. Действительно, математическое ожидание предсказания ансамбля $\mathbb{E}[\bar{f}] = \frac{1}{B}\sum \mathbb{E}[\hat{f}_b] = \mu$ равно математическому ожиданию предсказания одной модели. Поскольку базовые модели (полные деревья) обладают низким смещением, ансамбль сохраняет это свойство, одновременно радикально снижая дисперсию.

### 3. Бэггинг для классификации

В задачах классификации с $K$ классами бэггинг‑ансамбль обычно использует один из двух механизмов агрегирования:

- **Жёсткое голосование (hard voting):** каждый классификатор отдаёт голос за один класс, итоговое решение — класс, набравший большинство голосов.
- **Мягкое голосование (soft voting):** каждый классификатор возвращает оценку вероятности $p_k^{(b)}(\mathbf{x})$ для каждого класса $k$; итоговая вероятность вычисляется как среднее $\bar{p}_k(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^B p_k^{(b)}(\mathbf{x})$, после чего выбирается класс с максимальной вероятностью.

Мягкое голосование обычно предпочтительнее, так как учитывает «уверенность» каждой модели, что часто приводит к более высокому качеству.

Если классификаторы независимы и каждый ошибается с вероятностью $p < 1/2$, то вероятность ошибки ансамбля при жёстком голосовании даётся биномиальным хвостом:

$$
P_{\text{err}} = \sum_{k = \lfloor B/2 \rfloor + 1}^{B} \binom{B}{k} p^k (1-p)^{B-k}.
$$

Эта величина убывает экспоненциально с ростом $B$ (при $p < 1/2$), что теоретически обосновывает эффективность ансамблевого голосования. На практике корреляция между моделями несколько замедляет этот процесс, но общая тенденция сохраняется.

### 4. Out‑of‑Bag (OOB) оценка

Каждая бутстреп‑выборка $\mathcal{D}_b^*$ оставляет без использования около $36{,}8\%$ объектов исходной выборки. Эти объекты образуют **Out‑of‑Bag (OOB)** набор для $b$-й модели. Для каждого объекта $i$ можно получить OOB‑предсказание, усреднив (или проголосовав) только по тем моделям, в обучении которых этот объект не участвовал:

$$
\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i),
$$

где $\mathcal{B}_i = \{b \mid (\mathbf{x}_i, y_i) \notin \mathcal{D}_b^*\}$.

OOB‑ошибка, вычисленная на всех объектах, является практически несмещённой оценкой ошибки обобщения, сопоставимой по точности с кросс‑валидацией, но получаемой «бесплатно» в процессе обучения ансамбля. В отличие от обычной обучающей ошибки, OOB‑ошибка не страдает от оптимистического смещения, так как для каждого объекта используются только модели, не видевшие его при обучении.

### 5. Код и эксперименты

Проиллюстрируем свойства бэггинга на наборе данных `breast_cancer`. В качестве базовой модели возьмём полное решающее дерево (`max_depth=None`), которое характеризуется низким смещением и высокой дисперсией.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# Параметры эксперимента
B_range = [1, 3, 5, 10, 20, 50, 100, 200]
train_acc = []
test_acc = []
oob_acc = []

for B in B_range:
    # Bagging с полными деревьями, bootstrap=True, oob_score=True
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0),
        n_estimators=B,
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    bag.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, bag.predict(X_train)))
    test_acc.append(accuracy_score(y_test, bag.predict(X_test)))
    oob_acc.append(bag.oob_score_)

# Визуализация
plt.figure(figsize=(8, 5))
plt.plot(B_range, train_acc, 'bo-', label='train')
plt.plot(B_range, test_acc, 'ro-', label='test')
plt.plot(B_range, oob_acc, 'g^--', label='OOB')
plt.xscale('log')
plt.xlabel('Число моделей B')
plt.ylabel('Accuracy')
plt.title('Бэггинг с полными решающими деревьями (breast_cancer)')
plt.legend()
plt.grid(True)
plt.show()
```

График демонстрирует характерное поведение: с увеличением числа моделей тестовая точность возрастает и насыщается, обучающая точность остаётся близкой к единице, а OOB‑оценка практически совпадает с тестовой. Это подтверждает, что бэггинг эффективно уменьшает дисперсию, не увеличивая смещение, и что OOB‑ошибка служит надёжным индикатором обобщающей способности.

---

### Вопросы для самопроверки

#### Для начинающих
1. Зачем нужен бэггинг и какую проблему одиночных решающих деревьев он решает?
2. Что такое бутстреп‑выборка и почему в ней примерно $63\%$ уникальных объектов?
3. Почему OOB‑ошибка считается почти несмещённой оценкой качества обобщения?
4. Как бэггинг влияет на смещение и дисперсию базовой модели?

#### Для профессионалов
1. Докажите формулу $\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ для среднего коррелированных случайных величин с равными дисперсиями и попарной корреляцией $\rho$.
2. Объясните, почему бэггинг не уменьшает смещение. Приведите математическое обоснование с использованием линейности математического ожидания.
3. Сравните OOB‑ошибку и кросс‑валидацию с точки зрения смещения и дисперсии. В каких случаях OOB‑ошибка может давать смещённую оценку?
4. Выведите вероятность ошибки ансамбля при жёстком голосовании для $B$ независимых классификаторов, каждый из которых ошибается с вероятностью $p$, и покажите, что она экспоненциально убывает с ростом $B$ при $p < 1/2$.

## 9. Случайный лес: декорреляция деревьев, важность признаков и теория

Бэггинг над решающими деревьями снижает дисперсию, усредняя некоррелированные предсказания. Однако деревья, построенные по разным бутстреп-выборкам, всё ещё могут быть сильно скоррелированы, если в данных имеется несколько очень сильных признаков: каждое дерево будет использовать их в первых расщеплениях, что приводит к схожей структуре. Случайный лес (Random Forest), предложенный Брейманом (2001), решает эту проблему, вводя дополнительный источник случайности — при каждом расщеплении выбор признака ограничивается случайным подмножеством.

### 1. От бэггинга к случайному лесу

В случайном лесе сохраняется бутстреп-агрегирование базовых деревьев. Главное отличие заключается в модификации процедуры построения каждого дерева: в каждом узле перед расщеплением случайным образом отбирается $m$ признаков из полного набора $D$, и наилучшее расщепление ищется только среди этого подмножества. Типичные рекомендации:

- для классификации $m = \lfloor \sqrt{D} \rfloor$,
- для регрессии $m = \lfloor D/3 \rfloor$.

Новое дерево обучается без прунинга (до листьев с минимальным числом объектов), что сохраняет низкое смещение. Ансамблевое предсказание строится усреднением (регрессия) или голосованием (классификация) по всем деревьям.

Математическая мотивация для случайного выбора признаков — дальнейшее уменьшение средней попарной корреляции $\rho$ между предсказаниями деревьев. Как было показано в предыдущем разделе, дисперсия ансамбля равна

$$
\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
$$

Если несколько признаков доминируют, деревья в бэггинге будут похожи ($\rho$ велико). Ограничение числа кандидатов $m$ заставляет деревья использовать разные признаки, снижая $\rho$, а значит, и дисперсию ансамбля. При $m = D$ случайный лес вырождается в бэггинг над деревьями.

### 2. Анализ смещения и дисперсии для случайного леса

Случайный лес наследует от бэггинга свойство практически не увеличивать смещение по сравнению с одиночным несмещённым деревом (полное дерево имеет низкое смещение). С увеличением $m$ корреляция между деревьями растёт, дисперсия ансамбля увеличивается, а смещение может незначительно уменьшаться. С уменьшением $m$ корреляция падает, дисперсия снижается, но смещение может возрасти, если важные признаки систематически исключаются из рассмотрения, что ухудшает качество подгонки каждого отдельного дерева.

Оптимальное значение $m$ является компромиссом: оно достаточно мало, чтобы обеспечить декорреляцию, но достаточно велико, чтобы сохранить предсказательную силу каждого дерева. На практике стандартные эвристики $\sqrt{D}$ и $D/3$ работают хорошо, а точная настройка $m$ по валидационной выборке может дать дополнительный выигрыш.

### 3. Важность признаков в случайном лесе

Одно из главных достоинств случайного леса — возможность оценить вклад каждого признака в предсказание. Вводится две основные меры важности: основанная на уменьшении impurity (MDI) и основанная на перестановке (permutation importance).

#### 3.1. Уменьшение impurity (Mean Decrease in Impurity, MDI)

Для каждого признака $j$ суммируется уменьшение критерия impurity (например, Gini или энтропия) по всем узлам всех деревьев, в которых признак $j$ использовался для расщепления. Уменьшение взвешивается числом обучающих объектов, прошедших через узел. Формально, для одного дерева $T$:

$$
\text{FI}_j(T) = \sum_{t \in T: \text{split}(t) = j} \frac{m_t}{n} \,\Delta I_t,
$$

где $m_t$ — число объектов в узле $t$, $\Delta I_t = I_t - \frac{m_{t,L}}{m_t} I_{t,L} - \frac{m_{t,R}}{m_t} I_{t,R}$ — уменьшение impurity в узле. Затем усредняем по всем деревьям.

MDI вычислительно дёшева, так как получается «бесплатно» в процессе обучения. Однако она обладает смещением в пользу признаков с большим числом уникальных значений или категорий, поскольку такие признаки предоставляют больше возможностей для разбиений и могут искусственно завышать уменьшение impurity.

#### 3.2. Permutation Importance (важность по перестановке)

Идея предложена Брейманом (2001) и состоит в измерении падения качества предсказаний при разрушении связи между признаком и целевой переменной. Для каждого признака $j$ его значения случайным образом перемешиваются по наблюдениям (пермутация), после чего вычисляется ошибка модели на изменённых данных. Важность определяется как разность между ошибкой на перемешанных данных и исходной ошибкой, усреднённая по всем деревьям и обычно по нескольким случайным перестановкам:

$$
\text{FI}_j = \mathbb{E}_{\text{perm}}\bigl[ \text{Err}(\tilde{X}_j) - \text{Err}(X) \bigr],
$$

где $\tilde{X}_j$ — матрица признаков, в которой $j$-й столбец случайно переставлен.

Достоинство permutation importance — независимость от структуры модели и отсутствие смещения в сторону категорийных признаков. Однако для сильно коррелированных признаков перестановка одного из них не всегда полностью разрушает информацию (дублирующий признак может компенсировать), что иногда приводит к заниженным оценкам важности каждого из коррелированной группы.

#### Углублённый взгляд: связь MDI с объяснённой дисперсией

В регрессионном случайном лесе, использующем MSE в качестве impurity, суммарное уменьшение MSE по всем узлам леса обладает прозрачной вероятностной интерпретацией. Для одного дерева сумма $\sum_t \frac{m_t}{n} \Delta \text{MSE}_t$ равна разности между дисперсией отклика в корневом узле и средневзвешенной дисперсией в листьях — то есть доле дисперсии, «объяснённой» разбиениями. Усредняя по лесу, получаем, что MDI признака $j$ асимптотически (с ростом числа деревьев) оценивает вклад этого признака в объяснённую дисперсию. Более формально, если рассматривать случайный лес как адаптивную оценку функции регрессии, MDI можно связать с функциональным разложением ANOVA и оценками чувствительности (см. работы Штробля и др., 2007; Грёмпе, 2009). Это делает MDI особенно интерпретируемой мерой для регрессии, хотя для классификации интерпретация через уменьшение энтропии менее прямая, но сохраняет аналогичный смысл «уменьшения неопределённости».

### 4. Эксперименты

Продемонстрируем свойства случайного леса на наборе данных `breast_cancer`. Обучим лес из 500 деревьев с рекомендованным параметром $m = \lfloor\sqrt{D}\rfloor$, сравним с бэггингом и одиночным деревом, а также исследуем влияние $m$ на качество.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

D = X.shape[1]
m_default = int(np.sqrt(D))
print(f"Число признаков: {D}, рекомендованное m: {m_default}")

# Обучение моделей
rf = RandomForestClassifier(n_estimators=500, max_features=m_default,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=0),
    n_estimators=500, random_state=42, n_jobs=-1)
bag.fit(X_train, y_train)

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

# Сравнение точности
models = {'Одиночное дерево': tree, 'Бэггинг (500 деревьев)': bag,
          'Случайный лес (500 деревьев)': rf}
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: test accuracy = {acc:.4f}")
```

Ожидается, что случайный лес превзойдёт и одиночное дерево, и бэггинг благодаря дополнительной декорреляции.

#### Важность признаков (MDI и Permutation)

Извлечём MDI из обученного леса и рассчитаем permutation importance на тестовой выборке.

```python
# MDI (встроенная важность)
mdi_importances = rf.feature_importances_
indices_mdi = np.argsort(mdi_importances)[::-1]

# Permutation importance
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10,
                                     random_state=42, n_jobs=-1)
perm_importances = perm_result.importances_mean
indices_perm = np.argsort(perm_importances)[::-1]

# Визуализация топ-10 признаков по каждой мере
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(10), mdi_importances[indices_mdi][:10][::-1])
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([data.feature_names[i] for i in indices_mdi[:10][::-1]])
axes[0].set_title('MDI importance (топ-10)')

axes[1].barh(range(10), perm_importances[indices_perm][:10][::-1])
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([data.feature_names[i] for i in indices_perm[:10][::-1]])
axes[1].set_title('Permutation importance (топ-10)')

plt.tight_layout()
plt.show()
```

Две меры коррелируют, но могут давать разный порядок; permutation importance свободна от смещения MDI к многозначным признакам.

#### Зависимость точности от $m$

Зафиксируем $B=500$ и проварьируем `max_features` от 1 до $D$.

```python
m_values = np.arange(1, D+1)
test_acc = []
for m in m_values:
    rf_temp = RandomForestClassifier(n_estimators=500, max_features=m,
                                     random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, y_train)
    test_acc.append(accuracy_score(y_test, rf_temp.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(m_values, test_acc, 'o-')
plt.axvline(m_default, color='r', linestyle='--', label=f'default m={m_default}')
plt.xlabel('m (max_features)')
plt.ylabel('Test accuracy')
plt.title('Влияние числа случайных признаков на точность')
plt.legend()
plt.grid()
plt.show()
```

График покажет, что точность растёт до некоторого оптимального $m$, после чего может незначительно снизиться или выйти на плато. Рекомендованное значение, как правило, близко к оптимальному.

---

### Вопросы для самопроверки

#### Для начинающих
1. Чем случайный лес отличается от обычного бэггинга над решающими деревьями?
2. Зачем при построении деревьев в случайном лесе используется случайное подмножество признаков?
3. Как измерить важность признака в случайном лесе? Опишите идею MDI и Permutation Importance.
4. Что лучше: много глубоких деревьев или много неглубоких деревьев в случайном лесе, и почему?

#### Для профессионалов
1. Выведите формулу дисперсии ансамбля для среднего коррелированных предсказаний и покажите, как выбор $m$ влияет на корреляцию $\rho$ и, следовательно, на итоговую дисперсию.
2. Объясните причину смещения MDI в пользу признаков с большим числом категорий. Предложите способ коррекции этого смещения.
3. Сравните permutation importance и SHAP (SHapley Additive exPlanations) с точки зрения учёта корреляций между признаками и вычислительной сложности.
4. Как теоретически выбор $m$ влияет на смещение и дисперсию каждого отдельного дерева и всего ансамбля? Опишите компромисс «смещение–дисперсия» в контексте случайного леса.

## Градиентный бустинг: функциональный градиентный спуск и вывод для произвольных функций потерь

В отличие от бэггинга и случайного леса, которые строят ансамбль независимых моделей и усредняют их предсказания, градиентный бустинг (Friedman, 2001) формирует ансамбль последовательно: каждая следующая модель исправляет ошибки предыдущих. Математически этот процесс интерпретируется как функциональный градиентный спуск — оптимизация функции предсказания в функциональном пространстве.

### 1. Идея градиентного бустинга как функциональной оптимизации

Рассмотрим задачу обучения с учителем: по выборке $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ требуется найти функцию $F: \mathbb{R}^D \to \mathbb{R}$, минимизирующую эмпирический риск

$$
\mathcal{L}(F) = \sum_{i=1}^n L(y_i, F(\mathbf{x}_i)),
$$

где $L(y, F)$ — дифференцируемая по $F$ функция потерь. Мы ищем $F$ в виде аддитивной композиции $M$ слабых моделей (обычно неглубоких решающих деревьев):

$$
F_M(\mathbf{x}) = \sum_{m=0}^M \nu \, h_m(\mathbf{x}),
$$

$h_0(\mathbf{x})$ — начальное приближение (константа), $\nu > 0$ — темп обучения (learning rate). Ключевая идея: строить $h_m$ последовательно, на каждом шаге выбирая такую функцию, которая при добавлении к $F_{m-1}$ максимально уменьшает $\mathcal{L}$. Функциональный градиент определяет направление наискорейшего возрастания $\mathcal{L}$ в точке $F_{m-1}$; это функция

$$
\mathbf{x} \mapsto \left.\frac{\partial L(y, F(\mathbf{x}))}{\partial F}\right|_{F = F_{m-1}(\mathbf{x})}.
$$

Чтобы уменьшить потери, нужно добавить к $F_{m-1}$ функцию, пропорциональную отрицательному градиенту. Поскольку мы ограничены классом $\mathcal{H}$ (например, деревья фиксированной глубины), новую модель $h_m$ обучают так, чтобы она наилучшим образом приближала компоненты антиградиента на обучающих объектах.

### 2. Вывод псевдо‑остатков (обобщённый градиент)

Разложим $L(y_i, F_{m-1}(\mathbf{x}_i) + f(\mathbf{x}_i))$ по Тейлору первого порядка относительно $f$:

$$
L(y_i, F_{m-1}(\mathbf{x}_i) + f(\mathbf{x}_i)) \approx L(y_i, F_{m-1}(\mathbf{x}_i)) + g_i \, f(\mathbf{x}_i),
$$

где

$$
g_i = \left. \frac{\partial L(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}.
$$

Слагаемое $g_i f(\mathbf{x}_i)$ линейно по $f$. Чтобы максимально уменьшить лосс, следует выбрать $f$ пропорциональной вектору $(-g_1,\dots,-g_n)$ на обучающей выборке. Поэтому в качестве целевых значений для обучения $h_m$ принимают **псевдо‑остатки**

$$
r_{im} = -g_i.
$$

Рассмотрим основные функции потерь.

**Квадратичная ошибка (регрессия).** $L(y, F) = \frac{1}{2}(y - F)^2$. Тогда

$$
g_i = \frac{\partial L}{\partial F} = -(y_i - F_{m-1}(\mathbf{x}_i)),
$$

откуда $r_{im} = y_i - F_{m-1}(\mathbf{x}_i)$ — обычные остатки текущей модели.

**Log‑loss для бинарной классификации с метками $y \in \{-1, +1\}$.**  
Функция потерь имеет вид

$$
L(y, F) = \log\bigl(1 + \exp(-2yF)\bigr).
$$

Градиент по $F$:

$$
\begin{aligned}
\frac{\partial L}{\partial F}
&= \frac{1}{1 + \exp(-2yF)} \cdot \frac{\partial}{\partial F}\bigl(1 + \exp(-2yF)\bigr) \\
&= \frac{1}{1 + \exp(-2yF)} \cdot \bigl(-2y \exp(-2yF)\bigr) \\
&= \frac{-2y \exp(-2yF)}{1 + \exp(-2yF)}
= \frac{-2y}{1 + \exp(2yF)}.
\end{aligned}
$$

Следовательно,

$$
g_i = \frac{-2y_i}{1 + \exp\bigl(2y_i F_{m-1}(\mathbf{x}_i)\bigr)}, \qquad
r_{im} = -g_i = \frac{2y_i}{1 + \exp\bigl(2y_i F_{m-1}(\mathbf{x}_i)\bigr)}.
$$

Интерпретация: величина $y_i F_{m-1}(\mathbf{x}_i)$ — отступ (margin). Если отступ велик и положителен, знаменатель велик, $r_{im}$ мал — модель уверена в правильном ответе, корректировка почти не требуется. При отрицательном отступе $r_{im} \approx 2y_i$, что задаёт направление исправления.

**Логистическая регрессия с $y \in \{0, 1\}$ и кросс‑энтропийной функцией потерь.**  
Используется

$$
L(y, F) = -\bigl[y \log \sigma(F) + (1-y) \log(1 - \sigma(F))\bigr], \qquad \sigma(F) = \frac{1}{1 + e^{-F}}.
$$

Производная по $F$:

$$
\frac{\partial L}{\partial F} = \sigma(F) - y.
$$

Поэтому

$$
g_i = \sigma(F_{m-1}(\mathbf{x}_i)) - y_i, \qquad
r_{im} = y_i - \sigma(F_{m-1}(\mathbf{x}_i)),
$$

то есть разность между истинной меткой и предсказанной вероятностью класса 1.

Во всех случаях $h_m$ обучается как регрессионная модель, предсказывающая псевдо‑остатки $r_{im}$ по $\mathbf{x}_i$. После этого новая модель добавляется с весом $\nu$:

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

### 3. Выбор оптимального шага $\nu$ (learning rate)

Параметр $\nu$ управляет величиной шага функционального градиентного спуска. Малый фиксированный $\nu$ (shrinkage) замедляет обучение, но улучшает обобщающую способность, сглаживая вклад отдельных моделей. В пределе $\nu \to 0$ и $M \to \infty$ при $\nu M = \text{const}$ процесс аппроксимирует непрерывный градиентный спуск в функциональном пространстве, обладая регуляризирующим эффектом, сходным с $L_2$-штрафом.

Альтернативно, для каждого $m$ можно подбирать $\nu_m$ путём одномерной оптимизации (line search):

$$
\nu_m = \arg\min_{\nu} \sum_{i=1}^n L\bigl(y_i, F_{m-1}(\mathbf{x}_i) + \nu \, h_m(\mathbf{x}_i)\bigr).
$$

Такой подход используется, например, в классическом алгоритме AdaBoost для экспоненциальной функции потерь, где оптимальный вес имеет аналитическое выражение. В общем случае применяют численную оптимизацию (метод Ньютона или просто фиксируют малое $\nu$, компенсируя большим числом итераций).

### 4. Алгоритм градиентного бустинга (Generic Gradient Boosting)

Формально алгоритм выглядит следующим образом.

1. **Инициализация:**  
   $F_0(\mathbf{x}) = \arg\min_\gamma \sum_{i=1}^n L(y_i, \gamma)$.  
   Для квадратичной ошибки это среднее значение $y_i$; для логистической регрессии — логарифм отношения шансов $\log\frac{\sum y_i}{\sum (1-y_i)}$.

2. **Для $m = 1$ до $M$:**  
   1. Вычислить псевдо‑остатки:  
      $$ r_{im} = -\left. \frac{\partial L(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \quad i = 1,\dots,n. $$
   2. Обучить регрессионную модель $h_m$ (обычно дерево) на данных $\{(\mathbf{x}_i, r_{im})\}_{i=1}^n$.
   3. Найти шаг $\nu_m$ (line search или фиксированный $\nu$).
   4. Обновить модель:  
      $$ F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu_m \, h_m(\mathbf{x}). $$

3. **Выдать** $F_M(\mathbf{x})$ как итоговую модель.

#### Углублённый взгляд: Ньютоновский бустинг (второй порядок)

Сходимость можно улучшить, используя разложение второго порядка в точке $F_{m-1}(\mathbf{x}_i)$:

$$
L(y_i, F_{m-1} + f) \approx L(y_i, F_{m-1}) + g_i f + \frac{1}{2} h_i f^2,
$$

где

$$
g_i = \frac{\partial L(y_i, F)}{\partial F}\Big|_{F_{m-1}(\mathbf{x}_i)}, \qquad
h_i = \frac{\partial^2 L(y_i, F)}{\partial F^2}\Big|_{F_{m-1}(\mathbf{x}_i)}.
$$

При фиксированных $g_i$, $h_i$ квадратичная форма по $f$ достигает минимума при

$$
f = -\frac{g_i}{h_i}.
$$

Таким образом, в качестве целевых значений для листьев дерева можно брать величины $-g_i/h_i$, а само дерево строить с учётом весов $h_i$ (взвешенный метод наименьших квадратов). Именно этот подход применяется в XGBoost и LightGBM.

Выпишем $g_i$ и $h_i$ для основных функций потерь.

- **Квадратичная ошибка** $L = \tfrac{1}{2}(y - F)^2$:  
  $g_i = -(y_i - F_{m-1}(\mathbf{x}_i)), \quad h_i = 1$.  
  Оптимальное предсказание в листе: $f = y_i - F_{m-1}(\mathbf{x}_i)$ — средний остаток.

- **Log‑loss, $y \in \{-1, +1\}$** $L = \log(1 + \exp(-2yF))$:  
  Уже вычислено: $g_i = \frac{-2y_i}{1 + \exp(2y_i F_{m-1})}$. Вторая производная:  
  $$
  h_i = \frac{\partial g_i}{\partial F}
  = \frac{4 \exp(2y_i F_{m-1})}{\bigl(1 + \exp(2y_i F_{m-1})\bigr)^2}
  = 4 \, \sigma(2F_{m-1})\,\bigl(1 - \sigma(2F_{m-1})\bigr),
  $$
  где $\sigma(z) = 1/(1+e^{-z})$. Видно, что $h_i > 0$, что гарантирует выпуклость.

- **Логистическая регрессия, $y \in \{0,1\}$** $L = -[y\log\sigma(F) + (1-y)\log(1-\sigma(F))]$:  
  $g_i = \sigma(F_{m-1}) - y_i$,  
  $h_i = \sigma(F_{m-1})(1 - \sigma(F_{m-1}))$.

Ньютоновский шаг $-g_i/h_i$ для логистической регрессии даёт

$$
f = \frac{y_i - \sigma(F_{m-1})}{\sigma(F_{m-1})(1 - \sigma(F_{m-1}))},
$$

что в точности соответствует шагу метода Ньютона‑Рафсона для обычной логистической регрессии. Это позволяет бустингу быстрее достигать оптимума по сравнению с градиентным спуском первого порядка.

---

### Вопросы для самопроверки

#### Для начинающих

1. Что такое градиентный бустинг и в чём его принципиальное отличие от бэггинга и случайного леса?
2. Зачем нужен learning rate $\nu$ и как его выбор влияет на качество модели?
3. Что такое псевдо‑остатки? Как они вычисляются для квадратичной функции потерь и для логистической регрессии?
4. Почему градиентный бустинг склонен к переобучению и какие механизмы используют для его предотвращения?

#### Для профессионалов

1. Выведите псевдо‑остатки для log‑loss в обеих параметризациях ($y \in \{-1,+1\}$ и $y \in \{0,1\}$) и покажите их связь с остатками линейной регрессии.
2. Объясните, как shrinkage ($\nu < 1$) действует в качестве регуляризации. Сравните этот эффект с $L_2$-регуляризацией в линейных моделях.
3. Докажите, что градиентный бустинг является функциональным градиентным спуском в гильбертовом пространстве функций, порождённом ядром, задаваемым ансамблем деревьев.
4. Сравните сходимость градиентного бустинга первого порядка и Ньютоновского бустинга. В каких случаях предпочтителен каждый из них?

## Градиентный бустинг: реализация с нуля и сравнение с библиотеками

После теоретического обоснования градиентного бустинга как функционального градиентного спуска перейдём к его практической реализации. Мы напишем с нуля регрессионную и классификационную версии GBM, проверим их корректность на синтетических и реальных данных, исследуем влияние гиперпараметров и кратко познакомимся с основными промышленными библиотеками.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, log_loss
```

### 1. Реализация Gradient Boosting Machine (GBM) с нуля

Идея GBM заключается в последовательном добавлении слабых моделей (обычно неглубоких деревьев), каждая из которых обучается на псевдо‑остатках, равных антиградиенту функции потерь по текущему предсказанию. Мы реализуем два класса: для регрессии (квадратичная ошибка) и для бинарной классификации (логистическая функция потерь).

#### Регрессия

```python
class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None

    def fit(self, X, y):
        self.F0 = np.mean(y)                    # оптимальная константа для MSE
        F = np.full(len(y), self.F0)            # текущее предсказание
        self.trees = []
        for _ in range(self.n_estimators):
            residuals = y - F                   # отрицательный градиент для MSE
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X)
        return self

    def predict(self, X):
        F = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F
```

Здесь на каждом шаге вычисляются обычные остатки $y - F_{m-1}(\mathbf{x})$, и дерево учится их предсказывать. Добавление предсказаний дерева с малым шагом $\nu$ постепенно уменьшает ошибку.

#### Бинарная классификация (логистическая регрессия)

Для классификации используем сигмоидную функцию $\sigma(F) = 1/(1+e^{-F})$ и логарифмическую функцию потерь. Псевдо‑остатки равны $y_i - \sigma(F(\mathbf{x}_i))$.

```python
class GradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None

    @staticmethod
    def _sigmoid(F):
        return 1.0 / (1.0 + np.exp(-F))

    def fit(self, X, y):
        # доля положительного класса
        p = np.mean(y)
        self.F0 = np.log(p / (1 - p))           # оптимальная константа для log loss
        F = np.full(len(y), self.F0)
        self.trees = []
        for _ in range(self.n_estimators):
            p_pred = self._sigmoid(F)
            residuals = y - p_pred              # отрицательный градиент
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X)
        return self

    def predict_proba(self, X):
        F = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return self._sigmoid(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba >= 0.5).astype(int)
```

Инициализация $F_0 = \log\frac{\bar{p}}{1-\bar{p}}$ соответствует решению задачи $\arg\min_\gamma \sum_i L(y_i, \gamma)$ для логистической функции потерь.

### 2. Проверка корректности

Сравним нашу реализацию с `GradientBoostingRegressor` и `GradientBoostingClassifier` из `sklearn`, установив одинаковые параметры и убедившись в близости результатов.

#### Регрессия

```python
# Синтетические данные
np.random.seed(0)
n = 200
x = np.linspace(-3, 3, n)
y = x**2 + np.sin(5 * x) + np.random.normal(0, 0.3, n)
X = x.reshape(-1, 1)

# Разделение
X_train, X_test = X[:150], X[150:]
y_train, y_test = y[:150], y[150:]

# Наша модель
our_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
our_gbr.fit(X_train, y_train)
our_pred = our_gbr.predict(X_test)
our_mse = mean_squared_error(y_test, our_pred)

# sklearn
from sklearn.ensemble import GradientBoostingRegressor as SKGBR
sk_gbr = SKGBR(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
sk_gbr.fit(X_train, y_train)
sk_pred = sk_gbr.predict(X_test)
sk_mse = mean_squared_error(y_test, sk_pred)

print(f"Our GBR MSE: {our_mse:.4f}")
print(f"sklearn GBR MSE: {sk_mse:.4f}")
```

Визуализируем эволюцию предсказаний с ростом числа итераций:

```python
# Промежуточные предсказания для нескольких шагов
steps = [1, 5, 10, 50, 100]
plt.figure(figsize=(12, 6))
for i, step in enumerate(steps):
    model = GradientBoostingRegressor(n_estimators=step, learning_rate=0.1, max_depth=3)
    model.fit(X_train, y_train)
    y_plot = model.predict(x.reshape(-1, 1))
    plt.subplot(2, 3, i+1)
    plt.scatter(X_train, y_train, alpha=0.4)
    plt.plot(x, y_plot, 'r-')
    plt.title(f'n_estimators={step}')
plt.tight_layout()
plt.show()
```

Графики показывают, как простая константа постепенно превращается в сложную функцию, всё лучше приближая обучающие данные.

#### Классификация

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier as SKGBC

data = load_breast_cancer()
X_clf, y_clf = data.data, data.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42)

our_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
our_clf.fit(Xc_train, yc_train)
our_acc = accuracy_score(yc_test, our_clf.predict(Xc_test))
our_ll = log_loss(yc_test, our_clf.predict_proba(Xc_test))

sk_clf = SKGBC(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
sk_clf.fit(Xc_train, yc_train)
sk_acc = accuracy_score(yc_test, sk_clf.predict(Xc_test))
sk_ll = log_loss(yc_test, sk_clf.predict_proba(Xc_test))

print(f"Our GBC: accuracy={our_acc:.4f}, log_loss={our_ll:.4f}")
print(f"sklearn GBC: accuracy={sk_acc:.4f}, log_loss={sk_ll:.4f}")
```

Результаты будут близки, подтверждая корректность нашей реализации.

### 3. Анализ влияния learning rate и числа деревьев

Параметр `learning_rate` ($\nu$) контролирует вклад каждого дерева. При малых $\nu$ обучение замедляется, но, как правило, достигает лучшего обобщения при достаточно большом числе итераций. Проиллюстрируем это на регрессионном примере: зафиксируем `n_estimators=500` и будем варьировать $\nu$.

```python
learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
train_errors = []
test_errors = []
for lr in learning_rates:
    model = GradientBoostingRegressor(n_estimators=500, learning_rate=lr, max_depth=3)
    model.fit(X_train, y_train)
    train_errors.append(mean_squared_error(y_train, model.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, model.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(learning_rates, train_errors, 'bo-', label='train')
plt.plot(learning_rates, test_errors, 'rs-', label='test')
plt.xscale('log')
plt.xlabel('learning_rate')
plt.ylabel('MSE')
plt.legend()
plt.title('Влияние learning_rate при фиксированном числе итераций (500)')
plt.show()
```

График показывает, что при слишком большом $\nu$ модель переобучается (низкая ошибка на обучении, высокая на тесте). Уменьшение $\nu$ действует как регуляризация: тестовая ошибка сначала снижается, но при чрезмерно малом $\nu$ (и фиксированном $M=500$) может возрасти, так как модель не успевает сойтись. В реальных задачах число деревьев подбирают совместно с $\nu$.

### 4. Введение в библиотеки: XGBoost, LightGBM, CatBoost (краткий обзор)

Промышленные реализации градиентного бустинга — XGBoost, LightGBM и CatBoost — существенно оптимизируют базовый алгоритм, добавляя:

- учёт вторых производных (Ньютоновский бустинг),
- встроенную $L_1/L_2$ регуляризацию,
- эффективную работу с пропущенными значениями,
- специальную обработку категориальных признаков,
- параллельное построение деревьев и гистограммный поиск расщеплений.

Детальный разбор этих усовершенствований будет дан в следующих разделах. Здесь приведём лишь пример вызова для оценки скорости и качества на примере California Housing.

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import time

housing = fetch_california_housing()
Xh, yh = housing.data, housing.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.2, random_state=42)

def benchmark_model(name, model, Xt, yt, Xte, yte):
    start = time.time()
    model.fit(Xt, yt)
    train_time = time.time() - start
    pred = model.predict(Xte)
    mse = mean_squared_error(yte, pred)
    print(f"{name}: MSE={mse:.4f}, время обучения={train_time:.2f} сек")
    return mse, train_time

# Наша простая реализация
our_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4)
benchmark_model("Our GBM", our_gbr, Xh_train, yh_train, Xh_test, yh_test)

# XGBoost
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbosity=0)
    benchmark_model("XGBoost", xgb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("XGBoost не установлен")

# LightGBM
try:
    from lightgbm import LGBMRegressor
    lgb = LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbose=-1)
    benchmark_model("LightGBM", lgb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("LightGBM не установлен")

# CatBoost
try:
    from catboost import CatBoostRegressor
    ctb = CatBoostRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbose=0)
    benchmark_model("CatBoost", ctb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("CatBoost не установлен")
```

Скорость работы библиотек на порядки превосходит нашу наивную реализацию благодаря упомянутым выше алгоритмическим и инженерным оптимизациям.

---

### Вопросы для самопроверки

#### Для начинающих
1. Почему начальное предсказание $F_0$ в регрессионном GBM устанавливается равным среднему значению $y$?
2. Как в классификационном GBM вычисляются псевдо‑остатки и почему они отличаются от обычных остатков?
3. Каким образом малый learning rate помогает предотвратить переобучение? Приходится ли за это платить?

#### Для профессионалов
1. Объясните, почему в бинарной классификации с логарифмической функцией потерь начальное приближение берётся равным $\log\frac{\bar{p}}{1-\bar{p}}$. Выведите это значение из условия минимизации эмпирического риска.
2. Сравните вычислительную сложность нашей реализации и промышленных библиотек. За счёт каких механизмов XGBoost/LightGBM достигают высокой скорости?
3. Как можно ускорить вычисление псевдо‑остатков в нашей реализации, не меняя алгоритмической сути (подсказка: векторизация, предварительное выделение памяти)?

## Современный градиентный бустинг: XGBoost, LightGBM, CatBoost — алгоритмические инновации и математика

Развитие идей градиентного бустинга привело к появлению высокопроизводительных библиотек, которые не просто реализуют базовый алгоритм, но и вводят принципиальные усовершенствования: регуляризацию, использование вторых производных, специальные приёмы для работы с большими данными и категориальными признаками. В этом разделе мы разберём математический фундамент трёх наиболее влиятельных реализаций — XGBoost, LightGBM и CatBoost — и проведём их сравнительный анализ на реальной задаче.

### 1. XGBoost: регуляризация и Newton boosting

XGBoost (eXtreme Gradient Boosting), предложенный Ченом и Гестриным (Chen & Guestrin, 2016), строит аддитивную модель

$$
\hat{y}_i = \sum_{t=1}^M f_t(\mathbf{x}_i),\qquad f_t \in \mathcal{F},
$$

где $\mathcal{F}$ — пространство решающих деревьев. Главное нововведение — явная регуляризация сложности каждого дерева и использование вторых производных функции потерь (Newton boosting).

#### Целевая функция с регуляризацией

Полная целевая функция, минимизируемая на шаге $t$, имеет вид

$$
\text{Obj}^{(t)} = \sum_{i=1}^n L\bigl(y_i, \hat{y}_i^{(t-1)} + f_t(\mathbf{x}_i)\bigr) + \sum_{k=1}^t \Omega(f_k),
$$

где $\Omega(f_k)$ — штраф за сложность $k$-го дерева. Для одного дерева $f$ с $T$ листьями и вектором весов листьев $\mathbf{w} \in \mathbb{R}^T$ полагают

$$
\Omega(f) = \gamma T + \frac{1}{2}\lambda \|\mathbf{w}\|^2.
$$

Параметр $\gamma$ штрафует число листьев, а $\lambda$ — $L_2$-регуляризацию весов (обычно на практике добавляют и $L_1$-регуляризацию $\alpha \|\mathbf{w}\|_1$, но для простоты мы ограничимся $L_2$).

#### Ньютоновское разложение и оптимальные веса листьев

Разложим функцию потерь в точке $\hat{y}_i^{(t-1)}$ до второго порядка:

$$
L(y_i, \hat{y}_i^{(t-1)} + f_t(\mathbf{x}_i)) \approx L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t(\mathbf{x}_i)^2,
$$

где

$$
g_i = \left.\frac{\partial L(y_i, \hat{y})}{\partial \hat{y}}\right|_{\hat{y} = \hat{y}_i^{(t-1)}},\qquad
h_i = \left.\frac{\partial^2 L(y_i, \hat{y})}{\partial \hat{y}^2}\right|_{\hat{y} = \hat{y}_i^{(t-1)}}.
$$

Суммируя по всем объектам и добавляя регуляризацию, получаем приближённую целевую функцию для дерева $f_t$:

$$
\tilde{\text{Obj}}^{(t)} = \sum_{i=1}^n \left[g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t(\mathbf{x}_i)^2\right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2.
$$

Пусть дерево $f_t$ отображает каждый объект в один из $T$ листьев. Обозначим $I_j = \{i \mid \mathbf{x}_i \text{ попадает в лист } j\}$. Тогда $f_t(\mathbf{x}_i) = w_j$ для $i \in I_j$, и мы можем переписать сумму по листьям:

$$
\tilde{\text{Obj}}^{(t)} = \sum_{j=1}^T \left[ w_j \sum_{i \in I_j} g_i + \frac{1}{2} w_j^2 \sum_{i \in I_j} h_i \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2.
$$

При фиксированной структуре дерева (известны множества $I_j$) можно аналитически найти оптимальный вес каждого листа, минимизирующий квадратичную форму. Приравнивая производную по $w_j$ к нулю:

$$
\frac{\partial}{\partial w_j} = \sum_{i \in I_j} g_i + w_j \sum_{i \in I_j} h_i + \lambda w_j = 0 \;\Longrightarrow\;
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}.
$$

Подставляя $w_j^*$ обратно, получаем **оптимальное значение целевой функции** для данной структуры:

$$
\tilde{\text{Obj}}^* = -\frac{1}{2} \sum_{j=1}^T \frac{(\sum_{i \in I_j} g_i)^2}{\sum_{i \in I_j} h_i + \lambda} + \gamma T.
$$

Это выражение служит критерием качества разбиения — аналогом impurity в классическом CART. При поиске расщепления оценивается уменьшение $\tilde{\text{Obj}}^*$, называемое **gain**:

$$
\text{Gain} = \frac{1}{2}\left[ \frac{(\sum_{i \in I_L} g_i)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{(\sum_{i \in I_R} g_i)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{(\sum_{i \in I} g_i)^2}{\sum_{i \in I} h_i + \lambda} \right] - \gamma.
$$

Расщепление выполняется только если Gain > 0, что автоматически включает в себя pruning.

#### Вычислительные оптимизации

XGBoost использует специальную структуру данных (block structure) для хранения отсортированных значений каждого признака, позволяя быстро находить кандидаты на расщепление. Также реализовано эффективное обучение с пропущенными данными: для каждого узла выбирается направление «по умолчанию» (в левое или правое поддерево) на основе того, какое даёт меньший лосс. Поддержка разреженных форматов (CSR) и параллелизм на уровне построения деревьев делают XGBoost одной из самых масштабируемых реализаций.

### 2. LightGBM: GOSS и EFB

LightGBM (Light Gradient Boosting Machine), предложенный Ке и соавторами (Ke et al., 2017), решает проблему масштабирования градиентного бустинга на очень большие обучающие выборки и высокоразмерные пространства признаков. Два ключевых нововведения — Gradient‑based One‑Side Sampling (GOSS) и Exclusive Feature Bundling (EFB).

#### Gradient‑based One‑Side Sampling (GOSS)

Традиционный бустинг использует все объекты для вычисления градиентов, что дорого. Идея GOSS: объекты с большими градиентами (те, которые сильнее всего «ошибаются» и, следовательно, вносят основной вклад в обучение) должны сохраняться полностью, а из объектов с малыми градиентами можно взять случайную подвыборку, умножив их градиенты и гессианы на коэффициент для сохранения несмещённости оценки распределения данных.

Формально, пусть отсортированы модули градиентов: $|g_{(1)}| \le |g_{(2)}| \le \dots \le |g_{(n)}|$. Зададим долю $a$ (например, 0.2) для объектов с наибольшими градиентами и долю $b$ для случайной подвыборки из остальных. Тогда:

- Сохраняем все объекты с наибольшими градиентами: множество $A$ размера $a n$.
- Из оставшихся $(1-a)n$ объектов случайно выбираем подмножество $B$ размера $b n$ (где $b$ — доля от общего числа объектов).
- При вычислении сумм градиентов и гессианов для объектов из $B$ их величины умножаются на $\frac{1-a}{b}$.

Этот приём позволяет резко сократить объём данных для построения одного дерева, при этом оценка градиента остаётся асимптотически несмещённой, что доказано в оригинальной статье через анализ влияния на информацию Фишера.

#### Exclusive Feature Bundling (EFB)

В разреженных данных (например, после one‑hot кодирования) многие признаки являются взаимоисключающими (exclusive), то есть принимают ненулевые значения одновременно лишь для небольшого числа объектов. LightGBM объединяет такие признаки в один «пучок» (bundle), что уменьшает размерность без потери информации. Задача сводится к графовой раскраске: признаки — вершины, рёбра между невзаимоисключающими признаками. Для эффективного приближённого решения используется жадный алгоритм. После объединения каждый исходный признак кодируется смещёнными значениями в общем диапазоне, гарантируя различимость.

#### Leaf‑wise growth (рост по листьям)

В отличие от XGBoost и классического GBM, которые растят дерево по уровням (level‑wise), LightGBM выбирает лист с максимальным уменьшением потерь и расщепляет именно его (leaf‑wise). Такой подход порождает более глубокие и асимметричные деревья, которые при одинаковом числе листьев часто дают меньшую ошибку, но могут привести к переобучению на малых данных. Поэтому в LightGBM обязательно задаётся ограничение на максимальную глубину.

### 3. CatBoost: ordered boosting и обработка категориальных признаков

CatBoost (Categorical Boosting), разработанный Прохоровым и соавторами (Prokhorenkova et al., 2018), фокусируется на двух проблемах: надёжная обработка категориальных признаков и устранение систематического смещения градиентов (target leakage).

#### Проблема target leakage и ordered boosting

В стандартном градиентном бустинге при вычислении псевдо‑остатков и, особенно, при преобразовании категориальных признаков в числовые (target statistics) используется информация о целевой переменной текущего объекта, что ведёт к смещению оценок и переобучению (target leakage). CatBoost решает эту проблему с помощью **ordered boosting** — модифицированного алгоритма, который имитирует обучение на независимых данных.

Основная идея: генерируется случайная перестановка обучающих объектов. Для каждого объекта $i$ модель, используемая для вычисления псевдо‑остатков, обучается только на объектах, предшествующих $i$ в этой перестановке. Таким образом, для объекта $i$ получаем честный прогноз, не содержащий информации о $y_i$. На практике, для вычислительной эффективности, используется несколько перестановок и специальная схема «обучения без i-го объекта» (permutation‑based gradient boosting).

#### Обработка категориальных признаков: target statistics со сглаживанием

Для категориального признака CatBoost вычисляет **target statistics** — среднее значение целевой переменной по каждой категории, но с добавлением априорного сглаживания для борьбы с шумом при редких категориях. Формула для числового представления категории $k$:

$$
\hat{x}_k = \frac{\sum_{j=1}^n [x_j = k] \cdot y_j + a \cdot p_{\text{global}}}{n_k + a},
$$

где $n_k$ — число объектов с категорией $k$, $p_{\text{global}}$ — глобальное среднее целевой переменной (например, среднее $y$ для регрессии или базовая вероятность для классификации), $a > 0$ — параметр сглаживания, интерпретируемый как «виртуальные наблюдения» с известным средним. Эта формула выводится из байесовской оценки с априорным распределением, и аналогична сглаживанию Лапласа.

Важно, что эта статистика вычисляется с учётом порядка, то есть для объекта $i$ используются только предшествующие объекты в перестановке, что исключает утечку целевой переменной.

#### Симметричные деревья

CatBoost использует **симметричные деревья** (balanced trees): на каждом уровне все узлы используют один и тот же признак и порог для расщепления. Это упрощает и ускоряет как обучение, так и инференс (предсказание можно представить в виде быстрой табличной функции). Хотя симметричные деревья менее гибки, чем асимметричные, на практике они показывают конкурентоспособное качество, особенно в сочетании с ordered boosting.

### 4. Сравнительный эксперимент

Проведём сравнение трёх библиотек на наборе данных California Housing (задача регрессии). Для чистоты эксперимента фиксируем число итераций (1000), learning rate (0.05), максимальную глубину деревьев (6). Оценим время обучения и качество (RMSE).

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import time

# Импортируем библиотеки
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Загрузка данных
data = fetch_california_housing()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

models = {
    'XGBoost': XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                            verbosity=0, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=42),
    'CatBoost': CatBoostRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                                  verbose=0, random_seed=42)
}

results = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = {'RMSE': rmse, 'Time (s)': elapsed}
    print(f"{name:10s} | RMSE: {rmse:.4f} | Time: {elapsed:.2f} s")
```

Визуализируем результаты столбчатыми диаграммами.

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(results.keys())
rmse_vals = [results[n]['RMSE'] for n in names]
time_vals = [results[n]['Time (s)'] for n in names]

axes[0].bar(names, rmse_vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title('RMSE на тесте')
axes[0].set_ylabel('RMSE')

axes[1].bar(names, time_vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Время обучения')
axes[1].set_ylabel('Секунды')

plt.tight_layout()
plt.show()
```

Обычно LightGBM оказывается самым быстрым на больших наборах данных, CatBoost немного медленнее, но может давать лучший результат при наличии категориальных признаков, а XGBoost демонстрирует стабильно высокое качество за счёт сильной регуляризации. В данном эксперименте категориальных признаков нет, поэтому различия в качестве невелики, а время обучения отражает различия в реализации.

### Вопросы для самопроверки

#### Для начинающих
1. Назовите три самые популярные библиотеки градиентного бустинга и кратко охарактеризуйте их.
2. В чём отличие LightGBM от XGBoost по способу построения дерева (leaf‑wise vs level‑wise)?
3. Почему CatBoost хорошо работает с категориальными признаками без предварительного one‑hot кодирования?
4. Что такое early stopping и зачем он используется при обучении бустинга?

#### Для профессионалов
1. Выведите оптимальные веса листьев и значение целевой функции для XGBoost (формулы для $w_j^*$ и $\tilde{\text{Obj}}^*$). Покажите, как получается критерий Gain для расщепления.
2. Объясните, как GOSS корректирует смещение оценки градиента при подвыборке объектов с малыми градиентами. Зачем нужно умножать на коэффициент $(1-a)/b$?
3. Опишите алгоритм EFB и условия, при которых признаки могут быть объединены в один пучок без потери информации. Как задача упаковки признаков сводится к раскраске графа?
4. Сравните смещение и дисперсию моделей, получаемых при leaf‑wise и level‑wise росте деревьев. Почему leaf‑wise требует дополнительной регуляризации?

## Интерпретация ансамблей: SHAP, PDP, ICE и заключение главы

Ансамблевые методы, при всех их предсказательных достоинствах, часто воспринимаются как «чёрные ящики»: большое число деревьев делает модель непрозрачной. Однако современный инструментарий позволяет не только оценить глобальную важность признаков (раздел 9.3), но и ответить на вопрос, *почему* модель выдала конкретное предсказание и как признаки влияют на выход ансамбля в среднем. В этом заключительном разделе мы рассмотрим Partial Dependence Plots (PDP), Individual Conditional Expectation (ICE) и Shapley Additive Explanations (SHAP), а затем подведём итог всей главы, сравнив изученные методы и обозначив дальнейшие направления.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
import shap
```

### 1. Необходимость интерпретации и локальные объяснения

Важность признаков (MDI, permutation importance) даёт глобальную картину, но не объясняет отдельное предсказание и не показывает форму зависимости. В задачах, где требуется прозрачность решений (медицина, кредитный скоринг), нужны методы, вскрывающие внутреннюю логику ансамбля. Partial Dependence Plots описывают среднее маржинальное влияние признака, а SHAP-значения справедливо распределяют вклад каждого признака в конкретный прогноз, опираясь на кооперативную теорию игр.

### 2. Partial Dependence Plots (PDP) и Individual Conditional Expectation (ICE)

Пусть $S \subset \{1,\dots,D\}$ — подмножество признаков, а $\overline{S}$ — дополнение. Модель $\hat{f}$ обучена на $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$. Частичная зависимость для $S$ определяется как

$$
\hat{f}_S(\mathbf{x}_S) = \frac{1}{n} \sum_{i=1}^{n} \hat{f}\bigl(\mathbf{x}_S,\; \mathbf{x}_{\overline{S}}^{(i)}\bigr),
$$

где $\mathbf{x}_{\overline{S}}^{(i)}$ — значения остальных признаков из $i$-го обучающего наблюдения. На практике берут подвыборку, так как усреднение по всем $n$ объектам требует $n \cdot |\text{grid}|$ предсказаний. PDP показывает маржинальное влияние признаков $S$, усреднённое по распределению остальных признаков.

**ICE-кривые** (Individual Conditional Expectation) — это графики $\hat{f}(\mathbf{x}_S, \mathbf{x}_{\overline{S}}^{(i)})$ для каждого $i$ (или для случайной подвыборки). Они позволяют увидеть гетерогенность: различное влияние признака на разные объекты. Линия PDP получается усреднением ICE-кривых.

**Ограничения.** При сильной корреляции признаков усреднение может давать нереалистичные комбинации (например, рост и вес, противоречащие друг другу). В таких случаях полезна альтернатива — Accumulated Local Effects (ALE), которая усредняет локальные приращения предсказания, но в рамках данной главы мы ограничимся PDP/ICE.

**Пример.** Построим PDP и ICE для случайного леса на breast_cancer.

```python
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

features = [0, 1]  # mean radius, mean texture
fig, ax = plt.subplots(figsize=(8,5))
PartialDependenceDisplay.from_estimator(rf, X_train, features, ax=ax, kind='both', subsample=100)
plt.suptitle('PDP и ICE для двух признаков')
plt.show()
```

График показывает, как изменяется средняя предсказанная вероятность класса с ростом признака, и демонстрирует разброс индивидуальных кривых.

### 3. SHAP (SHapley Additive exPlanations)

SHAP (Lundberg & Lee, 2017) основан на значениях Шепли из теории кооперативных игр. Для одного наблюдения с вектором признаков $\mathbf{x}$ каждый признак рассматривается как игрок, а модель $\hat{f}$ — как характеристическая функция. Цель — справедливо распределить разницу $\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(X)]$ между признаками.

#### Определение Shapley value

Пусть $M$ — множество всех признаков ($|M| = D$). Для подмножества $S \subseteq M$ определим функцию ценности

$$
v(S) = \mathbb{E}\bigl[\hat{f}(X) \mid X_S = \mathbf{x}_S\bigr],
$$

то есть среднее предсказание модели при фиксированных значениях признаков из $S$ равными $\mathbf{x}_S$, а остальные признаки маргинализуются по распределению обучающей выборки. Тогда Shapley value для признака $j$:

$$
\phi_j(v) = \sum_{S \subseteq M \setminus \{j\}} \frac{|S|!\, (D - |S| - 1)!}{D!} \bigl[ v(S \cup \{j\}) - v(S) \bigr].
\tag{1}
$$

Интуитивно: $\phi_j$ — средневзвешенный маржинальный вклад признака $j$ при всех возможных порядках добавления признаков.

Свойства Шепли:
- **Эффективность:** $\sum_{j=1}^D \phi_j = \hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(X)]$.
- **Симметричность:** если $v(S \cup \{j\}) = v(S \cup \{k\})$ для всех $S$, то $\phi_j = \phi_k$.
- **Линейность:** для двух игр $v_1, v_2$ значение для $v_1+v_2$ равно сумме значений.
- **Фиктивный игрок:** если $v(S \cup \{j\}) = v(S)$ для всех $S$, то $\phi_j = 0$.

В контексте ML свойство эффективности особенно важно: сумма всех SHAP-значений для одного наблюдения в точности равна отклонению предсказания от среднего (базового значения), что даёт аддитивное объяснение.

#### Вычислительная сложность и TreeSHAP

Прямое вычисление (1) требует $2^D$ оценок, что невозможно для реальных задач. Для моделей на основе деревьев (в том числе ансамблей) разработан алгоритм **TreeSHAP**, вычисляющий точные Shapley values за полиномиальное время $O(T L D^2)$, где $T$ — число деревьев, $L$ — число листьев в дереве. Алгоритм рекурсивно обходит дерево, отслеживая число обучающих объектов, попадающих в каждый узел, и накапливает вклады, используя свойства покрытия.

> **Углублённый взгляд: идея TreeSHAP**
> В одном решающем дереве предсказание для $\mathbf{x}$ можно записать как сумму по листьям: $\hat{f}(\mathbf{x}) = \sum_{\ell} w_\ell \cdot \mathbf{1}_{\{\mathbf{x} \in R_\ell\}}$, где $w_\ell$ — значение в листе. Shapley value для признака $j$ выражается через взвешенную сумму по всем путям в дереве, где вес зависит от доли обучающих объектов, проходящих через каждое расщепление. Алгоритм вычисляет $\phi_j$ за один проход дерева, обновляя текущий вклад при каждом расщеплении по признаку $j$. Для ансамбля SHAP-значения усредняются по деревьям. Детали и доказательство корректности приведены в работе Lundberg et al. (2019).

### 4. Практическая визуализация SHAP

Используем библиотеку `shap` для анализа того же случайного леса на breast_cancer.

```python
explainer = shap.TreeExplainer(rf, X_train)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names, show=False)
plt.title('SHAP summary plot')
plt.tight_layout()
plt.show()
```

Summary plot отображает для каждого признака распределение SHAP-значений всех тестовых объектов, с цветовой кодировкой величины признака. Видно, какие признаки наиболее важны (по оси — средний абсолютный SHAP), а также направление влияния: высокие значения признака (красные точки) сдвигают предсказание в сторону класса 1 или 0.

Bar plot средней важности:

```python
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names, plot_type='bar', show=False)
plt.title('SHAP feature importance')
plt.tight_layout()
plt.show()
```

Waterfall plot для одного наблюдения (например, первого в тестовой выборке):

```python
shap.plots.waterfall(shap.Explanation(values=shap_values[1][0],
                     base_values=explainer.expected_value[1],
                     data=X_test[0], feature_names=data.feature_names))
```

Он показывает, как каждый признак отталкивает предсказание от базового значения (среднего) к итоговой вероятности.

Сравните SHAP-важности с MDI и permutation importance: SHAP обладает преимуществом согласованности (consistent), что делает его особенно надёжным.

### 5. Итоговое сравнение ансамблевых методов

Сведём характеристики основных методов в таблицу.

| Метод | Смещение | Дисперсия | Вычислительная сложность | Число гиперпараметров | Интерпретируемость |
|-------|----------|-----------|---------------------------|------------------------|---------------------|
| Одиночное дерево | Низкое | Высокая | $O(D \, n \log n)$ | Мало | Высокая (визуализация правил) |
| Бэггинг | Низкое | Снижена ($\rho\sigma^2$) | $O(B \cdot$ сложность дерева$)$ | Средне | Средняя (важность признаков) |
| Случайный лес | Низкое | Ещё ниже (декорреляция) | Как бэггинг, но с $m<D$ | Средне | Средняя (MDI + OOB) |
| Градиентный бустинг | Низкое (регулируется) | Низкая | $O(M \cdot$ сложность дерева$)$ | Много (learning rate, число итераций, глубина, регуляризация) | Низкая (SHAP восстанавливает) |

**Практические рекомендации:**
- Если нужен быстрый и неплохой baseline на табличных данных — случайный лес.
- Если требуется максимальная точность и есть ресурсы на настройку — градиентный бустинг (XGBoost/LightGBM/CatBoost).
- При большом количестве категориальных признаков особенно удобен CatBoost.
- Для сверхбольших выборок LightGBM предпочтителен по скорости.

### 6. Заключение и мост к глубокому обучению

Решающие деревья и их ансамбли остаются основным инструментом для табличных данных, где признаки имеют явный семантический смысл. Они не требуют масштабирования признаков, устойчивы к выбросам и способны улавливать нелинейные взаимодействия. Однако изображения, звук и текст содержат сырые данные высокой размерности, где решающие деревья неэффективны — там доминируют нейронные сети, автоматически извлекающие иерархические представления.

Современные исследования (TabNet, NODE) пытаются объединить интерпретируемость деревьев и мощь глубокого обучения, создавая дифференцируемые деревья и нейро-ансамбли. Таким образом, тема ансамблей не изолирована, а естественно вливается в более широкую картину машинного обучения.

---

### Вопросы для самопроверки (по всей главе)

#### Для начинающих
1. Чем решающее дерево принципиально отличается от линейной модели с точки зрения формы предсказаний?
2. Зачем случайный лес случайно выбирает признаки при каждом расщеплении?
3. В чём основная идея градиентного бустинга?
4. Назовите три популярные библиотеки градиентного бустинга.
5. Что такое SHAP-значения и какую информацию они дают о конкретном предсказании?
6. В какой ситуации вы выберете случайный лес, а в какой — градиентный бустинг?

#### Для профессионалов
1. Выведите формулу дисперсии ансамбля $\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ и объясните роль корреляции $\rho$.
2. Опишите, как алгоритм TreeSHAP вычисляет Shapley values за полиномиальное время, используя структуру дерева.
3. Сравните Gradient-based One-Side Sampling (GOSS) и обычную случайную подвыборку объектов: как GOSS сохраняет несмещённость оценки градиента?
4. Объясните механизм смещения предсказаний при вычислении target statistics в бустинге и как ordered boosting в CatBoost решает эту проблему.
5. Докажите, что SHAP-значения удовлетворяют аксиомам Шепли (эффективность, симметричность, линейность, фиктивный игрок).

---

### Задания повышенной сложности

1. **Собственная OOB-ошибка.** Реализуйте вычисление out-of-bag ошибки для случайного леса с нуля (без использования `oob_score_`). Сравните её с оценкой, полученной кросс-валидацией.
2. **Вывод оптимальных весов XGBoost.** Для заданной структуры дерева и целевой функции с регуляризацией $L_2$ выведите аналитически оптимальные веса листьев и значение критерия. Убедитесь, что они совпадают с формулами (14)–(15).
3. **SHAP-взаимодействия.** С помощью `shap.dependence_plot` постройте график зависимости SHAP-значения для одного признака, раскрашенный значениями другого признака. Интерпретируйте обнаруженные взаимодействия.
4. **Устойчивость к шумовым признакам.** Добавьте к данным 30% случайных признаков и сравните точность случайного леса и градиентного бустинга. Проанализируйте, какой из методов лучше отсеивает неинформативные признаки.
5. **Упрощённый TreeSHAP.** Для одного решающего дерева (не ансамбля) реализуйте вычисление Shapley values с нуля по рекурсивному алгоритму TreeSHAP. Проверьте на синтетическом примере, что сумма SHAP-значений равна разности предсказания и среднего.